# DeepVRegulome: Splice-Site Variant Scoring

This notebook demonstrates how to use DeepVRegulome's **splice-site models** to predict
the functional impact of variants at exon-intron junctions.

DeepVRegulome includes two splice-site models fine-tuned on GENCODE v41 annotations:

| Model | Description | Input |
|---|---|---|
| `splice_acceptor` | 3' splice site (AG dinucleotide at intron-exon boundary) | 90bp |
| `splice_donor` | 5' splice site (GT dinucleotide at exon-intron boundary) | 90bp |

**Key difference from TF/histone models:** Splice-site models use **90bp** input sequences
centered on the exon-intron junction, not 301bp.

**Prerequisites:**
- `pip install deepvregulome[all]`
- Human reference genome (hg38) FASTA + index (for coordinate-based scoring)

**Links:**
- [PyPI](https://pypi.org/project/deepvregulome/) |
  [HuggingFace Models](https://huggingface.co/duttaprat/DeepVRegulome) |
  [GitHub](https://github.com/DavuluriLab/DeepVRegulome) |
  [Paper (arXiv:2511.09026)](https://arxiv.org/abs/2511.09026)

## 1. Installation and Setup

In [19]:
# Install or upgrade (uncomment if needed)
# !pip install deepvregulome[all] --upgrade --quiet

In [20]:
import deepvregulome
print(f"DeepVRegulome version: {deepvregulome.__version__}")

DeepVRegulome version: 0.4.1


In [21]:
from deepvregulome import DVR
import pandas as pd

# ============================================================
# Initialize DVR
# ============================================================
# Option 1: With reference genome (enables coordinate-based scoring)
# dvr = DVR(genome="/path/to/hg38.fa")

# Option 2: Without reference genome (sequence-only scoring)
dvr = DVR()

print(dvr.registry)

ModelRegistry(458 TF + 4 histone + 2 splice = 464 models)


## 2. Explore Available Splice Models

In [22]:
# List all splice-site models
splice_models = dvr.list_models(model_type="SPLICE")

print(f"Number of splice models: {len(splice_models)}")
print()
for m in splice_models:
    print(f"  Name:         {m.name}")
    print(f"  Type:         {m.model_type}")
    print(f"  Input length: {m.input_length}bp")
    print(f"  HF subfolder: {m.subfolder}")
    print(f"  ROC-AUC:      {m.roc_auc}")
    print()

Number of splice models: 2

  Name:         splice_acceptor
  Type:         SPLICE
  Input length: 90bp
  HF subfolder: models/splice_acceptor
  ROC-AUC:      0.0

  Name:         splice_donor
  Type:         SPLICE
  Input length: 90bp
  HF subfolder: models/splice_donor
  ROC-AUC:      0.0



In [23]:
# You can also search for splice models by name
print(dvr.search_models("splice"))

['splice_acceptor', 'splice_donor']


In [24]:
# Compare with other model types
print(f"Total models:   {len(dvr.list_models())}")
print(f"TF models:      {len(dvr.list_models(model_type='TF'))}")
print(f"Histone models: {len(dvr.list_models(model_type='HISTONE'))}")
print(f"Splice models:  {len(dvr.list_models(model_type='SPLICE'))}")

Total models:   464
TF models:      458
Histone models: 4
Splice models:  2


## 3. Score a Single Splice-Site Variant (Sequence-Based)

If you already have the 90bp REF and ALT sequences centered on the splice junction,
you can score them directly using `score_sequence()`.

### Example: Variant in a splice acceptor region

Suppose you have identified a somatic variant at a known acceptor splice site.
The canonical acceptor dinucleotide is **AG** at the 3' end of the intron.
A mutation disrupting this AG can abolish splicing.

**Preparing 90bp sequences:**
1. Identify the exon-intron junction from GENCODE annotation
2. Extract 45bp upstream and 45bp downstream of the junction from hg38
3. The REF sequence is the wild-type; introduce your variant to create the ALT

In [25]:
# ============================================================
# Example: A>G mutation at a canonical acceptor AG dinucleotide
# ============================================================
# These are example 90bp sequences centered on an acceptor splice site.
# Replace with your actual sequences.

# Reference sequence (90bp) with intact acceptor AG
ref_seq_acceptor = (
    "GTAAGCTTTGGCAATCCTTTGAAAATGCTCTTCTTGCATTCTCTAG"  # 45bp (intron, ends with AG)
    "GCTACTGAATTCAAGCTTGGATCCAGATATCCCGGGAGCTCATCGA"  # 45bp (exon)
)

# Alternate sequence: A>G mutation disrupts the acceptor AG -> GG
alt_seq_acceptor = (
    "GTAAGCTTTGGCAATCCTTTGAAAATGCTCTTCTTGCATTCTCTGG"  # AG -> GG at position 44-45
    "GCTACTGAATTCAAGCTTGGATCCAGATATCCCGGGAGCTCATCGA"  # exon unchanged
)

print(f"REF length: {len(ref_seq_acceptor)}bp")
print(f"ALT length: {len(alt_seq_acceptor)}bp")
print(f"REF acceptor site: ...{ref_seq_acceptor[42:48]}...")
print(f"ALT acceptor site: ...{alt_seq_acceptor[42:48]}...")

REF length: 92bp
ALT length: 92bp
REF acceptor site: ...CTAGGC...
ALT acceptor site: ...CTGGGC...


In [29]:
# Score with the splice_acceptor model
result = dvr.score_sequence(
    ref_seq=ref_seq_acceptor,
    alt_seq=alt_seq_acceptor,
    models=["splice_acceptor"],
)

display(result)
print()
print("Columns: prob_ref = P(splice site | REF), prob_alt = P(splice site | ALT)")
print("A large negative log_odds_ratio indicates the variant disrupts the splice site.")

Tokenizing 1 REF + 1 ALT sequences using 1 processes...


Models:   0%|          | 0/1 [00:00<?, ?model/s]

,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,user,0,N,N,splice_acceptor,SPLICE,0.999999,0.999999,0.0,0.0



Columns: prob_ref = P(splice site | REF), prob_alt = P(splice site | ALT)
A large negative log_odds_ratio indicates the variant disrupts the splice site.


## 4. Score with Both Splice Models

You can score the same variant against both the acceptor and donor models
simultaneously. This is useful when the variant is near a junction and you
want to assess its impact on both splice signals.

In [30]:
# Score against both splice models at once
result_both = dvr.score_sequence(
    ref_seq=ref_seq_acceptor,
    alt_seq=alt_seq_acceptor,
    models=["splice_acceptor", "splice_donor"],
)

result_both

Tokenizing 1 REF + 1 ALT sequences using 1 processes...


Models:   0%|          | 0/2 [00:00<?, ?model/s]

,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,user,0,N,N,splice_acceptor,SPLICE,0.999999,0.999999,0.0000,0.0
1,user,0,N,N,splice_donor,SPLICE,0.001435,0.001616,-0.1718,0.0


In [31]:
# Or use model_type to score all splice models
result_all_splice = dvr.score_sequence(
    ref_seq=ref_seq_acceptor,
    alt_seq=alt_seq_acceptor,
    model_type="SPLICE",
)

result_all_splice

Tokenizing 1 REF + 1 ALT sequences using 1 processes...


Models:   0%|          | 0/2 [00:00<?, ?model/s]

,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,user,0,N,N,splice_acceptor,SPLICE,0.999999,0.999999,0.0000,0.0
1,user,0,N,N,splice_donor,SPLICE,0.001435,0.001616,-0.1718,0.0


## 5. Score a Splice Donor Variant

The canonical donor dinucleotide is **GT** at the 5' end of the intron.
Mutations disrupting this GT are among the most common causes of
aberrant splicing in disease.

In [32]:
# ============================================================
# Example: G>A mutation at a canonical donor GT dinucleotide
# ============================================================

# Reference sequence (90bp) with intact donor GT
ref_seq_donor = (
    "ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATC"  # 45bp (exon, placeholder)
    "GTAAGTTTCAATCCTTTGAAAATGCTCTTCTTGCATTCTCTAGGCTA"  # 45bp (intron, starts with GT)
)

# Alternate: G>A mutation disrupts the donor GT -> AT
alt_seq_donor = (
    "ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATC"  # exon unchanged
    "ATAAGTTTCAATCCTTTGAAAATGCTCTTCTTGCATTCTCTAGGCTA"  # GT -> AT at position 46-47
)

print(f"REF donor site: ...{ref_seq_donor[43:49]}...")
print(f"ALT donor site: ...{alt_seq_donor[43:49]}...")

REF donor site: ...GATCGT...
ALT donor site: ...GATCAT...


In [33]:
# Score with the splice_donor model
result_donor = dvr.score_sequence(
    ref_seq=ref_seq_donor,
    alt_seq=alt_seq_donor,
    models=["splice_donor"],
)

result_donor

Tokenizing 1 REF + 1 ALT sequences using 1 processes...


Models:   0%|          | 0/1 [00:00<?, ?model/s]

,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,user,0,N,N,splice_donor,SPLICE,0.020966,0.013046,0.696,-0.000166


## 6. Batch Scoring Multiple Splice Variants

If you have a list of variants at or near splice sites, you can extract
their 90bp sequences and score them in batch.

### Approach 1: From a DataFrame of pre-extracted sequences

In [34]:
# Suppose you have multiple splice-site variants with pre-extracted sequences
splice_variants = [
    {
        "variant_id": "chr1:12345:A>G",
        "junction_type": "acceptor",
        "ref_seq": ref_seq_acceptor,
        "alt_seq": alt_seq_acceptor,
    },
    {
        "variant_id": "chr7:98765:G>A",
        "junction_type": "donor",
        "ref_seq": ref_seq_donor,
        "alt_seq": alt_seq_donor,
    },
]

# Score each variant
all_results = []
for var in splice_variants:
    # Pick the appropriate model based on junction type
    model_name = f"splice_{var['junction_type']}"

    result = dvr.score_sequence(
        ref_seq=var["ref_seq"],
        alt_seq=var["alt_seq"],
        models=[model_name],
    )
    result["variant_id"] = var["variant_id"]
    result["junction_type"] = var["junction_type"]
    all_results.append(result)

# Combine results
df_results = pd.concat(all_results, ignore_index=True)
df_results

Tokenizing 1 REF + 1 ALT sequences using 1 processes...


Models:   0%|          | 0/1 [00:00<?, ?model/s]

Tokenizing 1 REF + 1 ALT sequences using 1 processes...


Models:   0%|          | 0/1 [00:00<?, ?model/s]

,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change,variant_id,junction_type
0,user,0,N,N,splice_acceptor,SPLICE,0.999999,0.999999,0.000,0.000000,chr1:12345:A>G,acceptor
1,user,0,N,N,splice_donor,SPLICE,0.020966,0.013046,0.696,-0.000166,chr7:98765:G>A,donor


### Approach 2: From genomic coordinates (requires reference genome)

If you have the variant coordinates and have initialized DVR with a reference
genome, you can use `score_variant()` directly.

In [35]:
# ============================================================
# Uncomment and run this cell if you have hg38.fa available
# ============================================================

# dvr_genome = DVR(genome="/path/to/hg38.fa")
#
# # Score a variant at a known splice acceptor site
# result = dvr_genome.score_variant(
#     chrom="chr17",
#     pos=41276045,          # Example: BRCA1 splice site region
#     ref="A",
#     alt="G",
#     models=["splice_acceptor"],
#     coordinate_system="1-based",
# )
# print(result)

## 7. Extracting 90bp Sequences from Genomic Coordinates

If you want to prepare sequences manually (e.g., for custom junction
definitions or non-standard annotations), here is how to extract
90bp sequences centered on a splice site using pysam.

In [36]:
# ============================================================
# Helper: extract 90bp splice-site sequences from hg38
# ============================================================
# Uncomment if you have pysam and hg38.fa available

# import pysam
#
# def extract_splice_sequences(
#     chrom: str,
#     junction_pos: int,   # 0-based position of the exon-intron boundary
#     ref_allele: str,
#     alt_allele: str,
#     fasta_path: str,
#     window: int = 90,
# ):
#     """
#     Extract 90bp REF and ALT sequences centered on a splice junction.
#
#     Parameters
#     ----------
#     chrom : str
#         Chromosome (e.g., 'chr17')
#     junction_pos : int
#         0-based position of the splice junction
#     ref_allele : str
#         Reference allele (single nucleotide for SNVs)
#     alt_allele : str
#         Alternate allele
#     fasta_path : str
#         Path to indexed hg38.fa
#     window : int
#         Total sequence length (default 90)
#
#     Returns
#     -------
#     ref_seq, alt_seq : str, str
#     """
#     half = window // 2
#     start = junction_pos - half
#     end = junction_pos + half
#
#     fasta = pysam.FastaFile(fasta_path)
#     ref_seq = fasta.fetch(chrom, start, end).upper()
#     fasta.close()
#
#     # Introduce the variant
#     var_offset = junction_pos - start
#     assert ref_seq[var_offset] == ref_allele, (
#         f"Reference mismatch: expected {ref_allele}, got {ref_seq[var_offset]}"
#     )
#     alt_seq = ref_seq[:var_offset] + alt_allele + ref_seq[var_offset + 1:]
#
#     return ref_seq, alt_seq
#
#
# # Example usage:
# ref, alt = extract_splice_sequences(
#     chrom="chr17",
#     junction_pos=41276044,    # 0-based
#     ref_allele="A",
#     alt_allele="G",
#     fasta_path="/path/to/hg38.fa",
# )
# print(f"REF: {ref[:20]}...{ref[-20:]}")
# print(f"ALT: {alt[:20]}...{alt[-20:]}")
# print(f"Length: {len(ref)}bp")

## 8. Interpreting Results

The output DataFrame contains these columns:

| Column | Description |
|---|---|
| `prob_ref` | Predicted probability of splice-site recognition for the REF sequence |
| `prob_alt` | Predicted probability of splice-site recognition for the ALT sequence |
| `log_odds_ratio` | log₂(prob_alt / (1 - prob_alt)) - log₂(prob_ref / (1 - prob_ref)) |
| `score_change` | prob_alt - prob_ref (raw probability difference) |

**Interpretation guide:**
- **Large negative `log_odds_ratio`:** The variant disrupts the splice site (the model
  predicts the ALT sequence is much less likely to be recognized as a splice site)
- **Near-zero `log_odds_ratio`:** The variant has minimal impact on splice-site recognition
- **Large positive `log_odds_ratio`:** The variant may create or strengthen a splice site
  (potential cryptic splice site activation)

In [37]:
# Quick interpretation of our results
for _, row in df_results.iterrows():
    direction = "DISRUPTS" if row["log_odds_ratio"] < -1 else (
        "STRENGTHENS" if row["log_odds_ratio"] > 1 else "NEUTRAL"
    )
    print(
        f"{row['variant_id']:20s} | "
        f"{row['junction_type']:8s} | "
        f"P(ref)={row['prob_ref']:.4f} -> P(alt)={row['prob_alt']:.4f} | "
        f"log_odds={row['log_odds_ratio']:+.3f} | "
        f"{direction}"
    )

chr1:12345:A>G       | acceptor | P(ref)=1.0000 -> P(alt)=1.0000 | log_odds=+0.000 | NEUTRAL
chr7:98765:G>A       | donor    | P(ref)=0.0210 -> P(alt)=0.0130 | log_odds=+0.696 | NEUTRAL


## 9. Combining Splice and TF Models

A variant near a splice site may also fall within a transcription factor
binding region. You can score the same variant against both splice and
TF models to get a comprehensive regulatory impact profile.

**Note:** TF models require 301bp sequences, while splice models require 90bp.
You need to extract the appropriate window for each model type.

In [38]:
# ============================================================
# Score the same genomic variant with both splice and TF models
# ============================================================
# This requires the reference genome to be available.
# Uncomment and adjust paths as needed.

# dvr_genome = DVR(genome="/path/to/hg38.fa")
#
# chrom = "chr17"
# pos = 41276045    # 1-based
# ref = "A"
# alt = "G"
#
# # Splice-site impact (90bp window)
# splice_result = dvr_genome.score_variant(
#     chrom=chrom, pos=pos, ref=ref, alt=alt,
#     models=["splice_acceptor", "splice_donor"],
# )
# print("=== Splice-site impact ===")
# print(splice_result)
# print()
#
# # TF-binding impact (301bp window)
# tf_result = dvr_genome.score_variant(
#     chrom=chrom, pos=pos, ref=ref, alt=alt,
#     models=["CTCFL", "SP1", "BRCA1"],
# )
# print("=== TF-binding impact ===")
# print(tf_result)
# print()
#
# # Combined view
# combined = pd.concat([splice_result, tf_result], ignore_index=True)
# print("=== Combined regulatory impact ===")
# print(combined[["model", "type", "prob_ref", "prob_alt", "log_odds_ratio", "score_change"]])

## 10. Loading Splice Models Directly (Without DVR Wrapper)

If you prefer to use the HuggingFace `transformers` library directly
(e.g., for custom pipelines or integration with other tools), you can
load the splice models from the HuggingFace Hub.

In [39]:
# ============================================================
# Direct model loading from HuggingFace
# ============================================================
# Uncomment to run (downloads model on first use, ~400MB per model)

# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# import torch
#
# # Load the splice acceptor model
# tokenizer = AutoTokenizer.from_pretrained(
#     "duttaprat/DeepVRegulome", subfolder="models/splice_acceptor"
# )
# model = AutoModelForSequenceClassification.from_pretrained(
#     "duttaprat/DeepVRegulome", subfolder="models/splice_acceptor"
# )
# model.eval()
#
# # DNABERT uses 6-mer tokenization
# # Convert a 90bp DNA sequence to space-separated 6-mers
# def seq_to_kmers(seq, k=6):
#     return " ".join([seq[i:i+k] for i in range(len(seq) - k + 1)])
#
# # Tokenize and predict
# kmer_seq = seq_to_kmers(ref_seq_acceptor)
# inputs = tokenizer(kmer_seq, return_tensors="pt", padding=True, truncation=True)
#
# with torch.no_grad():
#     outputs = model(**inputs)
#     probs = torch.softmax(outputs.logits, dim=-1)
#     print(f"P(not splice site): {probs[0][0]:.4f}")
#     print(f"P(splice site):     {probs[0][1]:.4f}")

## Summary

| Task | Method | Input |
|---|---|---|
| Score with pre-extracted sequences | `dvr.score_sequence(ref, alt, models=["splice_acceptor"])` | 90bp REF + ALT |
| Score by genomic coordinates | `dvr.score_variant(chrom, pos, ref, alt, models=["splice_acceptor"])` | Requires hg38.fa |
| Score all splice models at once | `dvr.score_sequence(ref, alt, model_type="SPLICE")` | 90bp REF + ALT |
| List splice models | `dvr.list_models(model_type="SPLICE")` | |
| Search by name | `dvr.search_models("splice")` | |
| Load directly from HuggingFace | `AutoModel.from_pretrained("duttaprat/DeepVRegulome", subfolder="models/splice_acceptor")` | |

For more information, see the [DeepVRegulome documentation](https://github.com/DavuluriLab/DeepVRegulome).